# PA State election results scraper HI!

This is a Python Notebook built to scrape the current PA election results page for specific county/counties. it riffs off [https://palewi.re/docs/first-github-scraper/](https://palewi.re/docs/first-github-scraper/), because it's a great resource for troubleshooting or creating more of scraper Notebooks as needs dictate.

### Notebook Steps

<details>
<summary>Table of Contents (click to expand)</summary>

1. [Env Setup](#env-setup)
1. [Import Python Libraries](#import-python-libraries)
1. [Check PA "Historical Elections Data"](#check-pa-historical-elections-data)
1. [Site-Change Detection](#site-change-detection)
1. [Get PA's list of all Elections available as Reports](#get-pas-list-of-all-elections-available-as-reports)
1. [Get an Election's CSV](#get-an-elections-csv)
1. [Lookup Election Details from List](#lookup-election-details-from-list)
1. [Recreate a human's report request](#recreate-a-humans-report-request)
1. [Make the request to generate a report](#make-the-request-to-generate-a-report)
1. [Save CSV to `data/raw_notebook/...`](#save-csv-to-dataraw_notebook)

### Env Setup

In [ ]:
# Setup env: imports and dependencies for the project
import os # for working with the operating system, like file paths
import sys # for checking system information
from pathlib import Path # for working with file paths

# add current directory to sys.path so we can import shared_setup
sys.path.insert(0, str(Path.cwd())) 

# import a shared tool to check dependencies
from shared_setup import check_dependencies
check_dependencies()

# icons
iWarn = "⚠️"
iOK = "✅"
iErr = "❌"

### Import Python Libraries

In [ ]:
import requests # for making HTTP requests
import csv # for working with CSV files
import json # for working with JSON data
import time # for adding delays between requests
import xml.etree.ElementTree as ET # for parsing XML responses
from datetime import datetime # for working with dates

# Create a session to maintain cookies across requests (important for WAF)
session = requests.Session()

# Set up realistic browser headers to avoid their bot-blocking WAF, Incapsula
# WAF = Web Application Firewall
session.headers.update({
    'User-Agent': 'PublicLedger newsgathering info@publicledger.news (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.9',
    'Accept-Encoding': 'gzip, deflate, br',
    'DNT': '1',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
    'Sec-Fetch-Dest': 'document',
    'Sec-Fetch-Mode': 'navigate',
    'Sec-Fetch-Site': 'none',
    'Sec-Fetch-User': '?1',
    'Cache-Control': 'max-age=0'
})

### Check PA "Historical Elections Data"

[https://www.electionreturns.pa.gov/ReportCenter/Reports](https://www.electionreturns.pa.gov/ReportCenter/Reports)

<details>
<summary>Instructions from their site (click to expand)</summary>

> Welcome to Pennsylvania's election reporting center.
> 
> To download results, click filters below to view election results based on your criteria. At any time, you may select, deselect or clear your filters.  
> 
> Step 1: Select an election.  
> Step 2: Select filters for the offices, candidate or party results you wish to download.  
> Step 3: Select a report type.  
> Step 4: Select a file export type.  
> Step 5: Generate your file!  
>
> Please note: In an ongoing effort to keep you informed of election night results, these pages contain data supplied by the counties in Pennsylvania. As this data is received on election night, our office will summarize and post the information to this site. It's possible these pages may contain unofficial results for several weeks after an election and should not be relied on as the official results until certified by the Secretary of the Commonwealth.

</details>

### Site-Change Detection 

Let's check that the page has not changed from our expectations. To do that, we need to set up our expected URLs. A `paSite` "root" url, then the various reporting-service "endpoints" they expose publicly that we will need to hit afterwards with requests.

In [ ]:
paSite = 'https://www.electionreturns.pa.gov/'
paReportService = paSite + 'ReportCenter/Reports'
paElectionList = paSite + 'api/Reports/GetElectionList'
paReport = paSite + 'api/Reports/GenerateReport'

First, we'll request the main URL and verify a few things are in its source code, to confirm the page has not changed. 

If it has changed, we will need to find out if it breaks this scraper and requires triage to adapt the code here to their new structure and features.

In [ ]:
# First, hit a top page in their API to establish cookies/session 
apiRes = session.get(
    paSite + 'api/ElectionReturn/', 
    timeout=30
)
print(f"API initial response: {apiRes.status_code}")

apiRes.raise_for_status()

# Check if we got blocked by WAF
if 'incapsula' in apiRes.text.lower() or 'incident_id' in apiRes.text.lower():
    print(f"{iWarn} WAF (Incapsula) detected - request was blocked")
    print("Try: 1) Running from different IP, 2) Adding longer delays, 3) Using browser automation")

# If not blocked, check if we got XML back (the API returns XML))
else:
    try:
        root = ET.fromstring(apiRes.text) # Parse XML response using ElementTree
        
        # Check if there's a namespace
        ns = {'ns': 'http://schemas.microsoft.com/2003/10/Serialization/Arrays'} if 'xmlns' in apiRes.text else {}
        
        if ns:
            stringElements = root.findall('.//ns:string', ns)
        else:
            stringElements = root.findall('.//string')
        
        print(f"{iOK} API returned XML with {len(stringElements)} tags")
    except ET.ParseError as e:
        print(f"{iWarn} Failed to parse XML: {e}")
        print(f"Response preview: {apiRes.text[:500]}...")

# check if tags match expectations: "<string>value1</string><string>value2</string>"
if stringElements and all(elem.text for elem in stringElements):
    print(f"{iOK} API results correct (contains <string> tags with text)")
else:
    print(f"{iWarn} API results look bad - check the response and parsing logic")
    exit(1) # quit the Notebook, since the rest will not work

In [ ]:
# top-level variables about Lancaster County, PA
targetCounty = 'Lancaster'
targetLegislativeDistrict = 111 # sorts state Special Elections
targetCountyFIPS = 42071 # not used yet! but worth saving
targetCountyID = 2325 # from https://www.electionreturns.pa.gov/ReportCenter/Reports source code

### Get PA's list of all Elections available as Reports

This next element will fetch the latest from the state print it out in a handy list.

In [ ]:
# Update session headers for API requests
session.headers.update({
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Cache-Control": "no-cache",
    "DNT": "1",
    "Origin": paSite,
    "Referer": paReportService,
    "Sec-Fetch-Dest": "empty",
    "Sec-Fetch-Mode": "cors",
    "Sec-Fetch-Site": "same-origin"
})

# Wait between requests to appear more human-like
time.sleep(2)

# Make request to get PA's list of Elections
print("Fetching election list from API...")
electionListResponse = session.get(paElectionList, timeout=30)
electionListResponse.raise_for_status()
status = iOK if electionListResponse.status_code == 200 else iWarn
print(f"{status} Election list response: {electionListResponse.status_code} {electionListResponse.reason}")

# API sometimes returns a JSON string, so normalize to a Python dict
electionList = electionListResponse.json()
if isinstance(electionList, str):
    electionList = json.loads(electionList)

if not isinstance(electionList, dict):
    raise TypeError(f"Expected dict, got {type(electionList)}")

table = electionList.get("Table", [])
status = iOK if table else iWarn
print(f"{status} Number of elections in 'Table': {len(table)}")
if table:
    print(f"Election row headers: {list(table[0].keys())}")

#### Print out all known elections

In [ ]:
import glob
from shared_table_display import display_markdown_table, sanitize_markdown_cell

# table columns to display, in the exact order requested
columns = [
    'Collected',
    'Electionid',
    'ElectionName',
    'ElectionDate',
    'ElectionType',
    'RegisteredVotes',
    'BallotsCounted',
]

def check_collected(row):
    """Return 🟢 if CSV exists, ⚫ if exists but starts with x_ (no data), ❌ if not collected."""
    try:
        date_str = row.get('ElectionDate', '')
        date_obj = datetime.strptime(date_str.split()[0], "%m/%d/%Y")
        formatted_date = date_obj.strftime("%Y-%m-%d")
        year = date_obj.year
    except (ValueError, AttributeError):
        return '❌'

    election_id = row.get('Electionid', '')
    elex_sub_type = row.get('ElectionType', '*')

    base_dir = os.path.join(
        '..', 'data', 'raw_notebook_csvs', 'state',
        targetCounty, str(year), str(elex_sub_type),
        str(formatted_date)
    )
    
    # Check for both regular and x_ prefixed files
    pattern = os.path.join(base_dir, f"{election_id}-*.csv")
    x_pattern = os.path.join(base_dir, f"x_{election_id}-*.csv")
    
    matches = glob.glob(pattern)
    x_matches = glob.glob(x_pattern)
    
    if x_matches:
        return '⚫'
    elif matches:
        return '🟢'
    else:
        return '❌'

# keep only recent elections and select only requested columns
rows_to_display = [
    {**{column: row.get(column, '') for column in columns if column != 'Collected'}, 'Collected': check_collected(row)}
    for row in table
    if row.get('ElectionYear', 0) >= 2012
]

# sort by ElectionType (G & P first, S for target district, other S, then O/others), then by Electionid descending
def election_sort_key(row):
    elex_type = row.get('ElectionType', '')
    elex_id = row.get('Electionid', 0)
    district = row.get('DistrictId', None)
    
    # G and P get priority 0
    if elex_type in ('G', 'P'):
        type_priority = 0
    # S elections for our target legislative district get priority 1
    elif elex_type == 'S' and district == targetLegislativeDistrict:
        type_priority = 1
    # Other S elections get priority 2
    elif elex_type == 'S':
        type_priority = 2
    # O and everything else gets priority 3
    else:
        type_priority = 3
    
    # Return tuple: (type_priority, -elex_id) for descending Electionid within groups
    return (type_priority, -elex_id)

rows_to_display = sorted(rows_to_display, key=election_sort_key)

# Display the table using shared display function
markdown_table = display_markdown_table(rows_to_display, columns)

# export table above to data/raw_notebook_csvs/state/Lancaster/STATE_README.md
export_path = Path('..') / 'data' / 'raw_notebook_csvs' / 'state' / 'Lancaster' / 'STATE_README.md'
with open(export_path, 'w', encoding='utf-8') as f:
    f.write(markdown_table)

### Get a Specific Election's CSV

We'll use the list of elections we already gathered to populate some details their service needs.

In [ ]:
electionId = 69 # ID chosen from list above
# G = General, P = Primary, S = Special, O = Off-Year
allowedElectionSubTypes = {'G', 'P', 'S', 'O'}
electionType = "S"

### Prepare to Fetch Election Details from List

In [ ]:
# lookup election ### from what we extracted above and save its data for use later
selectedElection = next(
    (row for row in table if row['Electionid'] == electionId and row['ElectionType'] == electionType),
    None,   
)

# throw an error if we couldn't find the election, to avoid making a bad request
if selectedElection is None:
    raise ValueError(f'Could not find election {electionId} with type {electionType}')


elexSubType = selectedElection['ElectionType']
elexDate = selectedElection['ElectionDate']
elexName = selectedElection['ElectionName']

# throw an error if we encounter an unexpected election type, to avoid making a bad request to the server
if elexSubType not in allowedElectionSubTypes:
    raise ValueError(f'Unexpected election type: {elexSubType}')

print(f"{iOK} Selected election #{electionId}: {elexName} ({elexDate})")

### Recreate a human's report request

Now we'll set up the arguments a human generates by using their various drop-downs and options.

In [ ]:
# this replicates a human manipulating the page to set various options
requestPayload = {
    # CountyIds is a list, but we only want one county
    "CountyIds": [targetCountyID],
    "ElectionID": electionId,
    "ElectionsubType": elexSubType, 
    "ReportType": "C", # County, 'D' = District, 'S' = State
    "ExportType": "C", # CSV or 'E' = Excel, 'T' = txt
    "FileName": "Official", # 'UnOfficial', 'Official'

    # empty means 'Select All'
    "OfficeIds":[],
    "RetOfficeIds":[],
    "PartyIds":[],
    "DistrictIds":[],
    "CandidateIds":[],
    "ReferendumIds":[],
    "ReferendumDetailIds":[],
}

### Make the request to generate a report

This recreates a submission to their server for a specific election report, for our given county, election and the file format we want back. 

In [ ]:
# Update headers for POST request
session.headers.update({
    'Accept': 'application/json, text/plain, */*',
    'Content-Type': 'application/json',
    'Origin': paSite,
    'Referer': paReportService
})

# Make request to get this specific report
print("Requesting report from API...")
reportCsv = session.post(
    paReport,
    data=json.dumps(requestPayload),
    timeout=30
)

if reportCsv.status_code == 200:
    print(f"{iOK} Report CSV request successful")
    # Uncomment if you wnat to see the raw CSV text (first 500 characters)
    # print(f"CSV sample: {reportCsv.text[:500]}...")
else:
    print(f"{iWarn} CSV request {reportCsv.status_code}'d - {targetCounty} may have no results for this election")
    print(f"{iWarn} Will save an empty placeholder CSV to record the attempt")


### Save CSV to `data/raw_notebook/...`


In [ ]:
# Parse the date string and format it as YYYY-MM-DD for filename
# API returns dates like "5/18/2025 12:00:00 AM" or similar format
date_obj = datetime.strptime(selectedElection["ElectionDate"].split()[0], "%m/%d/%Y")
elexDate = date_obj.strftime("%Y-%m-%d")

# get just the year
elexYear = date_obj.year

# "slugify" the election name to make it safe for filenames
elexNameSlug = elexName.lower().replace(' ', '-').replace('/', '-') # simple slugify for filename

# add a marker for "now", the point of running the scraper
nowish = datetime.now().isoformat(timespec="seconds")

# base directory for this election
saveDir = os.path.join(
    '..', 
    'data', 
    'raw_notebook_csvs', 
    'state',
    str(targetCounty), 
    str(elexYear), 
    str(elexSubType),
    str(elexDate),
)

# if year dir doesn't exist, make it
os.makedirs(saveDir, exist_ok=True)

hasData = reportCsv.status_code == 200

# Use x_ prefix for files with no data (404 or other non-200 responses)
prefix = "" if hasData else "x_"
baseFilename = f"{prefix}{electionId}-{elexNameSlug}.csv"

# if the base filename already exists, append a datestamp to avoid overwriting it
if os.path.exists(os.path.join(saveDir, baseFilename)):
    filename = f"{prefix}{electionId}-{elexNameSlug}-{nowish}.csv"
else:
    filename = baseFilename

filePath = os.path.join(saveDir, filename)

# save csv to a file for later analysis
with open(filePath, 'w', newline='') as outfile:
    writer = csv.writer(outfile)

    # add comments at the top of the CSV with election info
    comments = [
        f"Election: {elexName} ({elexDate}) - Election ID: {electionId} - {targetCounty} County",
        f"Report generated on: {nowish}",
    ]
    if not hasData:
        comments.append(f"NOTE: No data returned (HTTP {reportCsv.status_code}) - placeholder file only")
    for comment in comments:
        outfile.write(f"# {comment}\n")

    # write data rows only if the request succeeded; otherwise leave the file empty as a placeholder
    if hasData:
        reader = csv.reader(reportCsv.text.splitlines())
        for row in reader:
            writer.writerow(row)

status = iOK if hasData else iWarn
print(f"{status} Saved {'data' if hasData else 'empty placeholder'} to {filePath}")
